# Phase 6.1: Environment Initialization and Data Flash-Extraction
Mounts Google Drive to get checkpoints and extracts the raw dataset zip file to local Colab storage.

In [1]:
import os
import shutil
from google.colab import drive

# Mount Google Drive for persistent state tracking storage
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Define raw archive path and fast local runtime target destination paths
# DATASET_ZIP = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/BraTS2020_TrainingData.zip"
DATASET_ZIP = "/content/drive/MyDrive/ML-Datasets/BraTS2020_TrainingData_128.zip"
LOCAL_EXTRACT_DIR = "/content/MICCAI_BraTS2020_TrainingData_128"
# LOCAL_DATA_DIR = os.path.join(LOCAL_EXTRACT_DIR, "MICCAI_BraTS2020_TrainingData_128")
LOCAL_DATA_DIR = LOCAL_EXTRACT_DIR

# Verify archive existence immediately before runtime allocation
assert os.path.exists(DATASET_ZIP), f"Dataset archive not found: {DATASET_ZIP}"

# Robust extraction guard: triggers if directory does not exist or is completely empty
# if not os.path.exists(LOCAL_EXTRACT_DIR) or len(os.listdir(LOCAL_EXTRACT_DIR)) == 0:
if not os.path.exists(LOCAL_DATA_DIR) or len(os.listdir(LOCAL_DATA_DIR)) == 0:
    print(f"Extracting preprocessed dataset to fast local runtime storage: {LOCAL_EXTRACT_DIR}...")
    os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)
    shutil.unpack_archive(DATASET_ZIP, LOCAL_EXTRACT_DIR, "zip")
    print("Extraction complete. Preprocessed dataset ready for I/O operations.")
else:
    print("Valid local preprocessed dataset cache detected. Skipping extraction.")

Extracting preprocessed dataset to fast local runtime storage: /content/MICCAI_BraTS2020_TrainingData_128...
Extraction complete. Preprocessed dataset ready for I/O operations.


# Component Integration and Framework Imports

In [3]:
!pip install -q monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 45.6 MB/s eta 0:00:00


In [4]:
import sys
import random
import numpy as np
import torch
import torch.optim as optim
import itertools

# Enforce strict scientific reproducibility thresholds across packages
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Optimize CUDA runtime convolution algorithm selection for static tensor patches
torch.backends.cudnn.benchmark = True

# Add src to path
PROJECT_ROOT = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor"
sys.path.append(PROJECT_ROOT)

import src.config as config
from src.dataset import get_brats_dataloaders
from src.losses import MultiTaskLoss
from src.engine import run_training

# Instantiate the 5 distinct sub-network modules forming the end-to-end multi-task architecture
from src.models.mamba_backbone import MambaBackbone
from src.models.fusion import PresenceAwareCrossModalFusion
from src.models.mamba_backbone import SharedDeepMambaBackbone
from src.models.decoder import SegmentationDecoder3D
from src.models.classification import MorphologyGuidedClassifier

# Execution

In [5]:
import importlib

importlib.reload(config)

import src.engine as engine
importlib.reload(engine)

import src.models.mamba_backbone as mbb
importlib.reload(mbb)

import src.models.fusion as mf
importlib.reload(mf)

<module 'src.models.fusion' from '/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/src/models/fusion.py'>

In [6]:
from torch.amp import GradScaler

# Hardware runtime verification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Operational Hardware target identified: {device}")
print(f"Target Checkpoint Saving Directory: {config.CHECKPOINT_DIR}")

# 1. Pipeline Dataset Loaders Construction
print("Instantiating MONAI dictionary data pipelines...")
train_loader, val_loader = get_brats_dataloaders()

# 2. Structural Module Instantiations
print("Initializing neural net components...")
backbone = MambaBackbone(embed_dim=config.EMBED_DIM).to(device)
fusion = PresenceAwareCrossModalFusion(embed_dim=config.EMBED_DIM).to(device)
shared_backbone = SharedDeepMambaBackbone(embed_dim=config.EMBED_DIM).to(device)
decoder = SegmentationDecoder3D(embed_dim=config.EMBED_DIM, out_channels=config.NUM_SEG_CLASSES).to(device)
classifier = MorphologyGuidedClassifier(embed_dim=config.EMBED_DIM, num_classes=config.NUM_CLASS_CLASSES).to(device)

model_components = (backbone, fusion, shared_backbone, decoder, classifier)

# Compute total parameter profile summary metrics for publication tracking
total_params = sum(p.numel() for model in model_components for p in model.parameters())
print(f"Total Multi-Task Trainable Network Parameters: {total_params:,}")

# 3. Unified Multi-Task Optimization System Setup
criterion = MultiTaskLoss().to(device)

# Chain layer parameters
all_parameters = itertools.chain(
    backbone.parameters(),
    fusion.parameters(),
    shared_backbone.parameters(),
    decoder.parameters(),
    classifier.parameters()
)

optimizer = optim.AdamW(
    all_parameters,
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

# Automatically syncs decay frequency to match relative incremental epochs run window
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    # T_0=config.NUM_EPOCHS,
    T_0=config.TOTAL_EPOCHS, # Currently set to 150
    T_mult=1,
    eta_min=config.ETA_MIN
)

# Device-aware mixed-precision gradient scaling framework
scaler = GradScaler(enabled=(device.type == "cuda"))

# 4. Trigger Orchestration Engine Pipeline
print("Handing execution loop management over to stateful runtime engine...")
run_training(
    model_components=model_components,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    device=device
)

Operational Hardware target identified: cuda
Target Checkpoint Saving Directory: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints
Instantiating MONAI dictionary data pipelines...
Initializing neural net components...
Total Multi-Task Trainable Network Parameters: 5,143,543
Handing execution loop management over to stateful runtime engine...
[*] Found existing checkpoint record at: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/latest_checkpoint.pth. Loading state...
[+] Recovery complete. Resuming from absolute internal epoch counter: 115
[*] Incremental Run Configuration: Training from Epoch 115 -> Target Epoch 140 (+25 epochs)

--- Epoch 116/140 ---


[Train] Seg Loss: 0.4052 | Cls Loss: 0.1909 | Total Loss: 0.5961


[Val] Segmentation -> Mean Dice: 0.7961 (WT: 0.8635, TC: 0.7778, ET: 0.7469)
[Val] Classification -> Macro F1: 0.9582 | ROC-AUC: 0.9905
[Val] Combined Score: 0.9030
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/latest_checkpoint.pth

--- Epoch 117/140 ---


[Train] Seg Loss: 0.4177 | Cls Loss: 0.2207 | Total Loss: 0.6383


[Val] Segmentation -> Mean Dice: 0.7926 (WT: 0.8572, TC: 0.7748, ET: 0.7458)
[Val] Classification -> Macro F1: 0.8600 | ROC-AUC: 0.9571
[Val] Combined Score: 0.8622
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/latest_checkpoint.pth

--- Epoch 118/140 ---


...

[Val] Segmentation -> Mean Dice: 0.7977 (WT: 0.8629, TC: 0.7845, ET: 0.7456)
[Val] Classification -> Macro F1: 0.8600 | ROC-AUC: 0.9571
[Val] Combined Score: 0.8642
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/latest_checkpoint.pth

--- Epoch 139/140 ---


[Train] Seg Loss: 0.4466 | Cls Loss: 0.3035 | Total Loss: 0.7501


[Val] Segmentation -> Mean Dice: 0.7983 (WT: 0.8633, TC: 0.7848, ET: 0.7468)
[Val] Classification -> Macro F1: 0.8238 | ROC-AUC: 0.9714
[Val] Combined Score: 0.8579
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/latest_checkpoint.pth

--- Epoch 140/140 ---


[Train] Seg Loss: 0.4284 | Cls Loss: 0.2451 | Total Loss: 0.6735


[Val] Segmentation -> Mean Dice: 0.7983 (WT: 0.8630, TC: 0.7849, ET: 0.7471)
[Val] Classification -> Macro F1: 0.8600 | ROC-AUC: 0.9762
[Val] Combined Score: 0.8702
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/latest_checkpoint.pth

 Incremental cycle finished successfully. Total absolute epochs processed: 140
